In [ ]:
import cvxpy as cp
import numpy as np

# --- Helper Functions (unchanged) ---
def partial_trace_one_qubit(rho, trace_qubit, num_qubits):
    """Partial trace over one qubit (CVXPY-compatible)."""
    full_dim = 2 ** num_qubits
    reduced_dim = 2 ** (num_qubits - 1)
    reduced_rho = [[0 for _ in range(reduced_dim)] for _ in range(reduced_dim)]
    for i in range(full_dim):
        for j in range(full_dim):
            bi = [(i >> k) & 1 for k in reversed(range(num_qubits))]
            bj = [(j >> k) & 1 for k in reversed(range(num_qubits))]
            if bi[trace_qubit] == bj[trace_qubit]:
                bi_reduced = bi[:trace_qubit] + bi[trace_qubit+1:]
                bj_reduced = bj[:trace_qubit] + bj[trace_qubit+1:]
                row = sum([bit << (num_qubits - 2 - k) for k, bit in enumerate(bi_reduced)])
                col = sum([bit << (num_qubits - 2 - k) for k, bit in enumerate(bj_reduced)])
                reduced_rho[row][col] += rho[i, j]
    return cp.vstack([cp.hstack(row) for row in reduced_rho])

def partial_transpose_qubits(rho, transpose_qubits, num_qubits):
    """Partial transpose (CVXPY-compatible)."""
    dim = 2 ** num_qubits
    rho_pt = [[0 for _ in range(dim)] for _ in range(dim)]
    for i in range(dim):
        for j in range(dim):
            bi = [(i >> k) & 1 for k in reversed(range(num_qubits))]
            bj = [(j >> k) & 1 for k in reversed(range(num_qubits))]
            bi_new, bj_new = bi.copy(), bj.copy()
            for q in transpose_qubits:
                bi_new[q], bj_new[q] = bj[q], bi[q]
            row_new = sum([bit << (num_qubits - 1 - k) for k, bit in enumerate(bi_new)])
            col_new = sum([bit << (num_qubits - 1 - k) for k, bit in enumerate(bj_new)])
            rho_pt[row_new][col_new] += rho[i, j]
    return cp.vstack([cp.hstack(row) for row in rho_pt])

# --- Parameters ---
p = 0.75  # Parameter for rho_34
# --- p=(3\lambda+1)/4 ---
# --- p = 0.75 corresponds to \lambda=2/3 ---
# --- for \lambda=2/5 set p = 0.55 ---
epsilon = 0    # Slack for constraints
K = 10         # Number of M_i operators
q = 0.1   # Parameter for rho_12
rho_12 = np.array([
    [0.5, 0, 0, np.sqrt(1 - q)/2],
    [0, q/2, 0, 0],
    [0, 0, 0, 0],
    [np.sqrt(1 - q)/2, 0, 0, (1-q)/2]
]) #np.sqrt(1 - q)/2

sigma_34 = np.array([
    [(1+2*p)/6, 0, 0, (-1+4*p)/6],
    [0, (1-p)/3, 0, 0],
    [0, 0, (1-p)/3, 0],
    [(-1+4*p)/6, 0, 0, (1+2*p)/6]
])

rho_sigma = np.kron(rho_12, sigma_34)  # 16x16

    # --- Variables ---
M = [cp.Variable((16, 16), hermitian=True) for _ in range(K)]
identity_16 = np.eye(16)
identity_2 = np.eye(2)

    # --- Compute p = sum_i Tr[(Tr4(M_i) ⊗ I_2) @ rho_sigma] ---
sum_Tr4_Mi_tensor_I2 = 0
for M_i in M:
    Tr4_Mi = partial_trace_one_qubit(M_i, trace_qubit=3, num_qubits=4)  # Trace out subsystem 4
    Tr4_Mi_tensor_I2 = cp.kron(Tr4_Mi, identity_2)
    sum_Tr4_Mi_tensor_I2 += Tr4_Mi_tensor_I2
p = 2 * cp.real(cp.trace(sum_Tr4_Mi_tensor_I2 @ rho_sigma))

    # --- Objective ---
objective_terms = sum(cp.real(cp.trace(M_i @ rho_sigma)) for M_i in M)
objective = cp.Maximize((1 - p)/2 + objective_terms)

    # --- Constraints ---
constraints = [
    2 * cp.real(sum_Tr4_Mi_tensor_I2) <= identity_16,  # Constraint 1: sum_i (Tr4(M_i) ⊗ I_2) ≤ I_16 / 2
]
for M_i in M:
    constraints.extend([
        M_i >> 0,                              # Constraint 2: M_i ⪰ 0
        M_i << (1 - epsilon) * identity_16,    # Constraint 2: M_i ≤ (1 - ε)I
        partial_transpose_qubits(M_i, [1, 2], 4) >> 0  # Constraint 3: M_i^{T_23} ⪰ 0
])

    # --- Solve ---
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.MOSEK, verbose=True)
print("Optimal value:", problem.value)
print("p value", p.value)
print("Objective:", objective_terms.value)
    
